In [1]:
from transformers import BertModel, AutoTokenizer
from datasets import load_dataset
import torch
import gensim.downloader
import pandas as pd
import spacy
from collections import defaultdict
import random
import json
import nltk
from nltk.corpus import wordnet as wn
from collections import Counter
from itertools import combinations

In [2]:
def load_data(path):
    dataset = load_dataset("wikitext", "wikitext-103-v1", split="train")

    df = pd.read_csv(path, 
                sep="\t", 
                names=["lemma", "forms", "fam", "transf"])
    return dataset, df

In [3]:
def clean_dataset(dataset):
    content = []
    for line in dataset["text"]:
        line = line.strip().lower()
        if line and not line.startswith("=") and len(line.split()) > 4:
            content.append(line)
    return content

In [4]:
def get_target_words(df, glove_voc):
    target = {
            word.strip().lower()
            for form_list in df["forms"].str.split(",")
            for word in form_list
            if word.strip().lower() in glove_voc
    }
    return target

In [5]:
def get_matched_sent(content, target, json_path=None, filename="sent_df"):
    if json_path:
        return pd.read_json(json_path)

    nlp = spacy.blank("en")
    nlp.add_pipe("sentencizer")
    
    matched = defaultdict(set)

    for doc in nlp.pipe(content, batch_size=2048, n_process=2):
        for sent in doc.sents:
            if len(sent) > 40:
                continue

            sent_text = sent.text
            tokens = {token.text for token in sent}

            for word in target.intersection(tokens):
                matched[word].add(sent_text)

    def tag_word(sents):
        n = len(sents)
        if n >= 50:
            return "sampled"
        elif n >= 10:
            return "low_freq"
        return "excluded"

    def sample_sents(sents):
        sents = list(sents)
        n = len(sents)
        if n >= 50:
            return random.sample(sents, 50)
        return sents

    random.seed(42)
    sent_df = pd.DataFrame(matched.items(),
                            columns=["word", "sents"])

    sent_df["tag"] = sent_df["sents"].apply(tag_word)
    sent_df["sents"] = sent_df["sents"].apply(sample_sents)

    sent_df.to_json(f"{filename}.json")
    print(f"{filename}.json created")
    return sent_df

In [8]:
dataset, df = load_data("/home/onyxia/work/morph_families.tsv")
glove = gensim.downloader.load("glove-wiki-gigaword-50")
glove_voc = set(glove.key_to_index.keys())


print("content..")
content = clean_dataset(dataset)
del dataset

print("target..")
target = get_target_words(df, glove_voc)
del df

print("matched..")
sent_df = get_matched_sent(content, target, json_path="sent_df_init.json")

content..
target..
matched..


In [ ]:
sent_df.sample()

In [ ]:
def get_synonyms():
    print("get synonyms..")
    synonyms = set()
    for synset in wn.all_synsets():
        lemmas = [l.name().lower().replace("_", " ") for l in synset.lemmas()]
        for l1, l2 in combinations(lemmas, 2):
            res = tuple(sorted((l1, l2)))
            synonyms.add(res)

    return list(synonyms)


def get_antonyms():
    print("get antonyms..")
    antonyms = set()
    for synset in wn.all_synsets():
        for lemma in synset.lemmas():
            word = lemma.name().lower().replace("_", " ")
            for ant in lemma.antonyms():
                word_ant = ant.name().lower().replace("_", " ")
                res = tuple(sorted((word, word_ant)))
                antonyms.add(res)

    return list(antonyms)


def filter_word(pair_set, glove_voc, word_counts, filename):
    print("filter words..")
    random.seed(42)
    filtered = []
    
    for l1, l2 in pair_set:
        if l1 in glove_voc and l2 in glove_voc:
            if word_counts[l1] >= 10 and word_counts[l2] >= 10:
                filtered.append((l1, l2))

    if len(filtered) > 200:
        filtered = random.sample(filtered, 200)
    else:
        print(f"{len(filtered)} pairs found")
    
    with open(f"/home/onyxia/work/filtered_{filename}.json", "w") as f:
        json.dump(filtered, f)
        
    return list(filtered)


nltk.download("wordnet")
word_counts = Counter()
for sentence in content:
    word_counts.update(sentence.split())

filtered_synonyms = filter_word(get_synonyms(), glove_voc, word_counts, filename="synonyms")
filtered_antonyms = filter_word(get_antonyms(), glove_voc, word_counts, filename="antonyms")

In [ ]:
syn_anto = set()
for l1, l2 in filtered_synonyms + filtered_antonyms:
    syn_anto.add(l1)
    syn_anto.add(l2)

syn_ant_sent = get_matched_sent(content, syn_anto)
corpus = pd.concat([sent_df, syn_ant_sent], ignore_index=True)
corpus = corpus.drop_duplicates(subset="word").reset_index(drop=True)

corpus.to_json("/home/onyxia/work/corpus.json")

In [15]:
with open("/home/onyxia/work/filtered_synonyms.json", "r") as f:
    synonyms = json.load(f)

with open("/home/onyxia/work/filtered_antonyms.json", "r") as f:
    antonyms = json.load(f)

corpus = pd.read_json("/home/onyxia/work/corpus.json")

In [10]:
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU")

CUDA: True
GPU: Tesla T4


In [22]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")
bert = BertModel.from_pretrained("bert-base-cased", output_hidden_states=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
bert.to(device)
bert.eval() # to remove dropout (seen in DL class)

def get_target_emb(sentence, target, layer):
    # gets an unique target embedding
    encoding = tokenizer(sentence, return_tensors="pt")

    word_ids = encoding.word_ids()
    # word_ids = [None, 0, 1, 2, 3, 3, 3, .., None]

    unique_ids = set(i for i in word_ids if i is not None)

    # finds the id of the target word in word_ids
    target_id = None
    for identifier in unique_ids: 
        start, end = encoding.word_to_chars(identifier)
        if sentence[start:end] == target:
            target_id = identifier
            break
    
    if target_id is None:
        return None
        
    # gets all tokens composing the target word
    target_ids = [idx for idx, identifier in enumerate(word_ids)\
                    if identifier == target_id]
    
    with torch.no_grad():
        output = bert(encoding["input_ids"].to(device))

    hidden_states = output.hidden_states
    return hidden_states[layer][0][target_ids].mean(dim=0)
    
appelle_12 = get_target_emb("je m'appelle axel", "appelle", layer=12)
print(appelle_12)
print(appelle_12.size())

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tensor([ 3.0987e-01,  4.6448e-01, -9.0180e-02,  5.2743e-01, -6.2815e-02,
         1.4239e-01, -7.2600e-01,  4.4599e-02,  7.0334e-01, -2.1715e-02,
        -2.8211e-01,  3.4745e-01, -5.4558e-01, -1.3833e-01, -1.8108e-01,
         2.6533e-01, -4.1527e-01, -5.2353e-01,  3.0497e-01, -2.7238e-01,
         7.6660e-02, -2.0138e-01, -2.0921e-01, -1.0955e-02,  2.1132e-01,
        -4.0233e-01,  1.6663e-01,  4.8559e-01, -3.4469e-01,  4.5150e-01,
         1.9458e-01,  1.3732e-01,  4.3596e-01,  1.7029e-01, -1.7305e-03,
         5.2517e-01, -8.5059e-02,  5.6671e-01, -5.5372e-01, -5.6764e-01,
        -5.4467e-03, -1.3073e-01, -8.3412e-01,  2.1629e-01,  1.6047e-01,
        -7.5917e-01,  2.9853e-01, -7.6715e-01, -3.6985e-01, -2.2733e-01,
        -1.5057e-01,  3.6525e-01,  9.2066e-02,  6.1718e-01,  6.4936e-03,
         2.1852e-01, -4.4535e-01,  2.6758e-01, -3.1806e-01,  6.9796e-01,
        -7.1398e-01,  4.4390e-01, -2.3305e-01,  1.6609e-01,  3.2848e-01,
         6.7812e-01, -1.7804e-01,  1.1203e-01, -1.8

In [ ]:
from tqdm import tqdm

# AI-helped part to solve how to
# save my embeddings and to know where
# to start again if there was a crash,
# as datalab stops often after one hour of service

for layer in [1, 6, 12]:
    print(f"layer {layer}..")
    
    checkpoint_path = f"/home/onyxia/work/embeddings_layer{layer}.json"
    try:
        with open(checkpoint_path, "r") as f:
            layer_embeddings = json.load(f)
        print(f"  resumed from checkpoint: {len(layer_embeddings)}")
    except FileNotFoundError:
        layer_embeddings = {}

    for _, row in tqdm(corpus.iterrows(), total=len(corpus)):
        word = row["word"]
        
        if word in layer_embeddings:
            continue
            
        sentences = row["sents"]
        embs = []
        for sent in sentences:
            emb = get_target_emb(sent, word, layer)
            if emb is not None:
                embs.append(emb)
        
        if embs:
            # I choose to get the mean contextual embeddings to
            # compare with static embeddings of glove
            layer_embeddings[word] = torch.stack(embs).mean(dim=0).cpu().tolist()

    with open(checkpoint_path, "w") as f:
        json.dump(layer_embeddings, f)
    print(f"  layer {layer} done: {len(layer_embeddings)} words")

try:
    with open("/home/onyxia/work/embeddings_glove.json", "r") as f:
        glove_embeddings = json.load(f)
except FileNotFoundError:
    glove_embeddings = {word: glove[word].tolist() 
                        for word in corpus["word"] if word in glove_voc}
    with open("/home/onyxia/work/embeddings_glove.json", "w") as f:
        json.dump(glove_embeddings, f)

100%|██████████| 938/938 [06:50<00:00,  2.29it/s]


  layer 1 done: 938 words
layer 6..


100%|██████████| 938/938 [06:47<00:00,  2.30it/s]


  layer 6 done: 938 words
layer 12..


100%|██████████| 938/938 [06:38<00:00,  2.35it/s]


  layer 12 done: 938 words


GloVe: [-0.13886000216007233, 1.1401000022888184, -0.8521199822425842, -0.2921200096607208, 0.7553399801254272, 0.8276200294494629, -0.3181000053882599, 0.007220400031656027, -0.34762001037597656, 1.073099970817566, -0.24664999544620514, 0.977649986743927, -0.5583500266075134, -0.09031800180673599, 0.831820011138916, -0.33316999673843384, 0.22648000717163086, 0.3091300129890442, 0.02692900039255619, -0.08673900365829468, -0.14702999591827393, 1.3543000221252441, 0.5369499921798706, 0.43735000491142273, 1.274899959564209, -1.4381999969482422, -1.281499981880188, -0.15196000039577484, 1.0506000518798828, -0.9364399909973145, 2.7560999393463135, 0.5896700024604797, -0.294730007648468, 0.27573999762535095, -0.32927998900413513, -0.20100000500679016, -0.28547000885009766, -0.45987001061439514, -0.146029993891716, -0.6937199831008911, 0.07076100260019302, -0.19325999915599823, -0.18549999594688416, -0.16095000505447388, 0.24267999827861786, 0.2078399956226349, 0.030923999845981598, -1.371099